# Extending NeuRosetta

Two ways to extend the toolbox without fighting it:

1. **User-level** — attach results as graph properties / metadata, batch with
   `Forest.apply`, persist in `.nr`. No package edits.
2. **Library-level** — new metric in `utils/` → wrap in `ops/` → bind on
   `Tree` / `Forest` → optional `describe` registry entry.

Read {doc}`architecture` first for the layer model. This notebook is mostly
**(1)** runnable end-to-end; **(2)** is a checklist with copy-paste skeletons.


In [1]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import neurosetta as nr

%matplotlib inline

tree = nr.load_example_data(nr.example_ids[0])
forest = nr.load_example_data()
tree.summary()


ID,nodes,branches,leaves,cable,units,isReduced,Flag
720575940596125868,2010,99,111,404.2,micron,False,False


## User-level: custom graph properties

`Tree` uses `__slots__`, so `tree.my_thing = …` fails. Put analysis outputs on
the **graph** instead — same mechanism NeuRosetta uses for `radius`,
`Path_length`, etc.

| Level | Meaning | Typical use |
|-------|---------|-------------|
| `"v"` | per vertex | node scores, labels, distances |
| `"e"` | per edge | custom lengths, weights |
| `"g"` | whole graph | scalar summaries you want bound |

`set_property(..., create=True, dtype=…)` creates the map if missing.
`dtype` is a graph-tool type string (`"double"`, `"int"`, `"bool"`, …).


In [2]:
# Euclidean distance of each node from the root (in current units)
root = tree.get_root_coordinate()
coords = tree.get_node_coordinates()
dist = np.linalg.norm(coords - root, axis=1)

tree.set_property(
    "dist_from_root",
    dist,
    level="v",
    create=True,
    dtype="double",
)

print(tree.has_property("dist_from_root", level="v"))
print(tree.get_property("dist_from_root")[:5])
print("listed among v props?", "dist_from_root" in tree.list_properties("v"))


True
[0.         0.16768326 0.346392   0.68085983 0.61251662]
listed among v props? True


### Metadata vs properties

- **Metadata** — small key/value bag (`Neuron_type`, your QC notes, …). Survives
  `.nr` save. Core keys (`units`, `file_path`, `isReduced`, `Flag`) are
  protected — use the dedicated setters.
- **Properties** — arrays aligned to vertices / edges / the graph object.

```python
tree.set_meta("pipeline", "dist_from_root_v1")
tree.get_meta("pipeline")
```


In [3]:
tree.set_meta("pipeline", "dist_from_root_v1")
print(tree.get_meta("pipeline"))
print([k for k in tree.metadata if k == "pipeline"])


dist_from_root_v1
['pipeline']


### Persist in `.nr`

Bound properties that graph-tool can pickle round-trip through
`Tree.save_tree` / `nr.load`. That is the point of native files for analysis
pipelines — results stay glued to the morphology.


In [4]:
out = Path(tempfile.mkdtemp())
path = out / f"{tree.ID}.nr"
tree.save_tree(path)

reloaded = nr.load(path)
print("has prop:", reloaded.has_property("dist_from_root", level="v"))
print("meta pipeline:", reloaded.get_meta("pipeline"))
print(reloaded.get_property("dist_from_root")[:5])


has prop: True
meta pipeline: dist_from_root_v1
[0.         0.16768326 0.346392   0.68085983 0.61251662]


## Reusable functions + `Forest.apply`

Wrap your analysis as a plain `Tree -> value` (or `Tree -> None` with
`bind=True` side effects). Then batch with `Forest.apply` — same parallel /
progress knobs as built-in forest metrics.


In [5]:
def mean_dist_from_root(t: nr.Tree) -> float:
    root = t.get_root_coordinate()
    coords = t.get_node_coordinates()
    return float(np.linalg.norm(coords - root, axis=1).mean())


values = forest.apply(mean_dist_from_root, parallel=False)
print(dict(zip(forest.ids(), np.round(values, 3))))


{720575940596125868: np.float64(35.136), 720575940599459782: np.float64(29.423), 720575940599704006: np.float64(44.479), 720575940599729862: np.float64(40.712)}


Bind-on-each-tree pattern (mutates members, returns `None`s):


In [6]:
def bind_dist_from_root(t: nr.Tree) -> None:
    root = t.get_root_coordinate()
    coords = t.get_node_coordinates()
    dist = np.linalg.norm(coords - root, axis=1)
    t.set_property(
        "dist_from_root",
        dist,
        level="v",
        create=True,
        dtype="double",
    )


forest.apply(bind_dist_from_root, parallel=False)
print(all(t.has_property("dist_from_root", level="v") for t in forest))


True


### Built-in descriptor tables

If what you need is already a registered metric, skip the custom property and
use {func}`~neurosetta.describe` — it orchestrates existing `Tree` / `Forest`
methods into a tidy table (see {doc}`../reference/metrics`):


In [7]:
df = nr.describe(forest, domains="topology", metrics=["count_nodes", "count_leaves"])
df


,neuron_id,topology.count_nodes,topology.count_leaves
0,720575940596125868,2010.0,111.0
1,720575940599459782,1408.0,99.0
2,720575940599704006,2043.0,156.0
3,720575940599729862,2299.0,166.0


## Library-level: adding a first-class op

Do this when the metric belongs in the package for everyone. Checklist:

| Step | Where | What |
|------|-------|------|
| 1 | `utils/graph_utils/` (or `geometry_utils/`) | Pure function on `Graph` / arrays — **no** `Tree` import |
| 2 | `ops/tree_graphs/your_module.py` | `def foo(tree: _Tree, …)` calling the utils helper; end with `enrich_tree_graph_docstrings(globals())` |
| 3 | `ops/tree_graphs/__init__.py` (+ `ops/__init__.py` if needed) | Re-export |
| 4 | `api/tree_class.py` | `foo = foo` (or alias) |
| 5 | `api/forest_class.py` | `foo = _forest_op(foo)` if batching makes sense |
| 6 | `neurosetta/__init__.py` | Optional public functional export |
| 7 | `utils/metrics/registry.py` | Optional — so `describe` / docs tables pick it up |
| 8 | `tests/…` | Unit test utils; round-trip test via `Tree` |

Dependency rule reminder: **`utils/` must not import `ops/` or `api/`**.

### Skeleton (illustrative — not executed here)

**utils** (`utils/graph_utils/my_metric.py`):

```python
from graph_tool.all import Graph
from numpy import ndarray

def node_abs_x(g: Graph) -> ndarray:
    return abs(g.vp["x"].a)
```

**ops** (`ops/tree_graphs/tree_my_metric.py`):

```python
from numpy import ndarray
from ...core import _Tree
from ...utils.graph_utils.my_metric import node_abs_x as _node_abs_x
from .._doc_helpers import enrich_tree_graph_docstrings

def get_abs_x(tree: _Tree, bind: bool = False) -> ndarray | None:
    """Absolute x-coordinate per node.

    Parameters
    ----------
    tree : _Tree
        Neuron tree.
    bind : bool, optional
        If True, store as vertex property ``abs_x`` and return None.
    """
    values = _node_abs_x(tree.graph)
    if bind:
        tree.set_property("abs_x", values, level="v", create=True, dtype="double")
        return None
    return values

enrich_tree_graph_docstrings(globals())
```

**api bind**:

```python
# tree_class.py
from ..ops.tree_graphs import get_abs_x
class Tree(_Tree):
    get_abs_x = get_abs_x

# forest_class.py
get_abs_x = _forest_op(get_abs_x)
```

Follow existing modules (`tree_counting.py` ↔ `utils/graph_utils/counting.py`)
as the template — including docstring style so automodules stay coherent.

### Naming aliases

If the public method name should differ from the op function
(`reduce_tree` → `Tree.get_reduced_tree`), add an entry to
`TREE_METHOD_ALIASES` in `ops/_doc_helpers.py` so See Also links resolve.


## Quick reference

| Goal | Approach |
|------|----------|
| One-off node/edge array on a tree | `tree.set_property(..., create=True)` |
| Small label / QC flag | `tree.set_meta(...)` / `set_flag` |
| Survive save/load | Write `.nr` |
| Batch custom fn | `forest.apply(fn, parallel=…)` |
| Table of built-in metrics | `nr.describe(tree_or_forest, …)` |
| New package metric | `utils` → `ops` → `Tree`/`Forest` bind → tests → optional registry |

Next: {doc}`../api/index` for the generated reference, or
{doc}`../reference/metrics` for the descriptor catalogue.
